Financial Runway Assistant Chatbot

In [ ]:
# 1. Imports
import os
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

In [ ]:
# 2. Load configuration
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")


In [ ]:
# 3. Create the client and select a model
openai = OpenAI(api_key=api_key)

MODEL = "gpt-4o-mini"

In [ ]:
# Define the Financial Runway Assistant's instructions and few-shot examples

system_message = """
You are a Financial Runway Assistant.

Gather information about the potential client's:
- income
- expenses
- cash savings
- investments
- debts
- financial goals

Ask no more than two questions at a time.
Do not calculate financial runway until the required information is available.
Do not invent missing information.

Before calculating financial runway, confirm:
- whether financial amounts are monthly or annual
- the client's financial goal
- whether investment assets are accessible or held in retirement accounts

After calculating the financial runway, connect the result to the client's
stated goal and explain the assumptions and limitations.

Example 1:
User: I earn $120,000 per year and have $50,000 saved.

Assistant: Thanks. I have your annual income and current savings. What are your
average monthly expenses, and do you have any investment assets?

Example 2:
User: I have $300,000 invested and spend about $6,000 per month.

Assistant: Your annual spending is approximately $72,000. Before estimating
your financial runway, do you also have cash savings or outstanding debts?

Example 3:
User: I make $10,000.

Assistant: Is that $10,000 per month or per year? I want to confirm the time
period before using it in the calculation.

Example 4:
User: I have $50,000 in cash, $300,000 invested, no debt, and spend $5,000
per month.

Assistant: Thanks. What financial goal would you like these assets to support?
Also, are your investments held in a taxable brokerage account, retirement
accounts, or both?
"""

In [ ]:
def chat(message, history):
    # Combine the system prompt, conversation history, and latest user message
    messages = (
        [{"role": "system", "content": system_message}]
        + history
        + [{"role": "user", "content": message}]
    )

In [ ]:
# Create the chat callback and stream the model's response

def chat(message, history):
    messages = (
        [{"role": "system", "content": system_message}]
        + history
        + [{"role": "user", "content": message}]
    )

    stream = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        stream=True
    )

    response = ""

    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        yield response

In [ ]:
# Create and launch the Financial Runway Assistant interface

gr.ChatInterface(
    fn=chat,
    type="messages",
    title="Financial Runway Assistant"
).launch()